# Lab 4 Report: Leakage-Safe Classification & Regression Pipelines

## 1. Problem Framing & Objectives
This report documents the design, validation, tuning, and final evaluation of two supervised machine learning pipelines for a customer support ticketing system:
- **Experiment A (Binary Classification):** Predict whether a support ticket will be `escalated` (`1` = Escalated, `0` = Not Escalated) using ticket attributes (`complexity_score`, `team_experience_years`, `priority`, `channel`, and `customer_tenure_months`). Due to class imbalance (709 non-escalated vs. 191 escalated tickets, ~21.2% positive class), **F1-score** is used as the primary evaluation metric to balance precision and recall.
- **Experiment B (Regression):** Predict continuous ticket resolution duration in hours (`resolution_time_hours`) using the same feature set. **Mean Absolute Error (MAE)** is used as the primary optimization metric, with **Root Mean Squared Error (RMSE)** additionally evaluated on the holdout test set to assess sensitivity to large errors and outliers.

In both experiments, the unique identifier `ticket_id` is excluded prior to modeling to prevent memorization and spurious correlations.

---
# Experiment A: Ticket Escalation Classification

## 2. Data Loading, Exploratory Inspection & Splitting Strategy
The classification dataset (`classification_escalation.csv`) contains **900 records** and **7 columns**, with missing values present across both numeric (`complexity_score`, `team_experience_years`, `customer_tenure_months`) and categorical (`priority`, `channel`) predictors.

To establish an unbiased evaluation benchmark, the dataset is split into **80% training (720 samples)** and **20% holdout test (180 samples)** sets using `stratify=y` and `random_state=42`. Stratification guarantees that the ~21.2% minority escalation rate is preserved identically across both partitions.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_validate,
    GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error
)

clf_df = pd.read_csv("classification_escalation.csv")
display(clf_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


In [2]:
print(clf_df.shape)
print(clf_df.info())
print(clf_df.isnull().sum())
print(clf_df["escalated"].value_counts())

(900, 7)
<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id               900 non-null    str    
 1   complexity_score        878 non-null    float64
 2   team_experience_years   864 non-null    float64
 3   priority                887 non-null    str    
 4   channel                 891 non-null    str    
 5   customer_tenure_months  869 non-null    float64
 6   escalated               900 non-null    int64  
dtypes: float64(3), int64(1), str(3)
memory usage: 49.3 KB
None
ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64
escalated
0    709
1    191
Name: count, dtype: int64


In [4]:
X = clf_df.drop(columns=["ticket_id", "escalated"])
y = clf_df["escalated"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 3. Preprocessing & Leakage Prevention
All feature transformations are encapsulated within a `ColumnTransformer` placed inside a `scikit-learn` `Pipeline`:
- **Numeric Features (`complexity_score`, `team_experience_years`, `customer_tenure_months`):** Missing values are imputed using the **median** (robust to skewness/outliers), followed by **StandardScaler** (zero mean, unit variance) so distance-based models (k-NN) and regularized linear models (Logistic Regression) treat features on a comparable scale.
- **Categorical Features (`priority`, `channel`):** Missing entries are imputed using the **most frequent category**, followed by **OneHotEncoder (`handle_unknown="ignore"`)** to convert nominal categories into binary indicators without failing on unseen categories at inference time.

**Leakage Prevention:** By nesting `preprocessor` inside the `Pipeline`, every imputation statistic (medians, modes), scaling parameter (means, standard deviations), and categorical vocabulary is fitted *strictly on each training fold* during cross-validation and never exposed to validation folds or the holdout test set.

In [8]:
num_cols = X_train.select_dtypes(include=np.number).columns
cat_cols = X_train.select_dtypes(exclude=np.number).columns

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

## 4. Validation Strategy & Candidate Model Comparison
We evaluate three distinct model families using identical **5-fold `StratifiedKFold` cross-validation** (`shuffle=True, random_state=42`) scored on **F1**:
1. **Logistic Regression (`max_iter=1000`):** Linear baseline estimating class log-odds.
2. **k-Nearest Neighbors (`KNeighborsClassifier`):** Non-parametric local distance classifier.
3. **Decision Tree (`DecisionTreeClassifier`, `random_state=42`):** Non-linear axis-aligned rule learner.

In [11]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring="f1")
    results.append({
        "Model": name,
        "Mean F1": scores["test_score"].mean(),
        "Std F1": scores["test_score"].std()
    })

classification_results = pd.DataFrame(results)
display(classification_results)

,Model,Mean F1,Std F1
0,Logistic Regression,0.392651,0.062747
1,KNN,0.360835,0.050295
2,Decision Tree,0.330666,0.058126


### CV Results Analysis & Hyperparameter Tuning
| Model | Mean CV F1 | Std CV F1 |
| :--- | :---: | :---: |
| **Logistic Regression** | **0.3927** | 0.0627 |
| k-Nearest Neighbors (k-NN) | 0.3608 | 0.0503 |
| Decision Tree | 0.3307 | 0.0581 |

**Logistic Regression** achieved the highest mean CV F1 (`0.3927`), outperforming k-NN and the unpruned Decision Tree (which tends to overfit small tabular datasets). We select Logistic Regression for hyperparameter tuning via `GridSearchCV` over the inverse regularization strength `model__C` $\in \{0.01, 0.1, 1, 10, 100\}$ using the exact same 5-fold stratified CV splits.

In [12]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

Best Parameters: {'model__C': 10}
Best CV F1: 0.4225932078257659


## 5. Final Test Evaluation & Error Analysis (Experiment A)
Tuning `C=10` relaxed L2 regularization and improved mean CV F1 from `0.3927` to **`0.4226`**. We now evaluate `grid.best_estimator_` once on the untouched holdout test set (`n = 180`).

In [13]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.96      0.89       142
           1       0.62      0.21      0.31        38

    accuracy                           0.81       180
   macro avg       0.72      0.59      0.60       180
weighted avg       0.78      0.81      0.77       180

[[137   5]
 [ 30   8]]


### Interpretation of Test Results, False Positives & False Negatives
- **Test Metrics:** Overall accuracy is **81%**, while minority class (`escalated = 1`) **Precision = 0.62**, **Recall = 0.21**, and **F1 = 0.31**.
- **Confusion Matrix Breakdown (`[[137, 5], [30, 8]]`):**
  - **True Negatives (137) & False Positives (5):** Only 5 non-escalated tickets were falsely flagged for escalation (low false-alarm rate of ~3.5%). In operations, false positives cause minor senior-agent triage overhead.
  - **True Positives (8) & False Negatives (30):** The model missed **30 out of 38 actual escalations** (78.9% false negative rate). Because the default `0.50` probability threshold favors the 78.8% majority class, the classifier is conservative.
- **Justification & Limitations:** Logistic Regression (`C=10`) is interpretable, fast, and achieved the strongest CV F1 without overfitting. However, its low recall under default thresholding is a significant operational limitation when missed escalations lead to SLA breaches. Future iterations should apply `class_weight="balanced"` or tune the decision threshold on validation probabilities.

---
# Experiment B: Resolution Time Regression

## 6. Data Loading, Preprocessing & Validation Strategy
The regression dataset (`resolution_time_regression.csv`) contains **800 records** and **7 columns**, with minor missingness in numeric columns (`complexity_score`: 16, `team_experience_years`: 24, `customer_tenure_months`: 20). Summary statistics show `resolution_time_hours` has a mean of **5.87 hours**, median of **5.39 hours**, IQR of **2.63 to 8.35 hours**, and a right-skewed maximum of **27.49 hours** (indicating high-duration outliers).

- **Splitting & Preprocessing:** After dropping `ticket_id`, the data is split 80/20 (`random_state=42`) into 640 training and 160 testing instances. An identical leakage-safe `ColumnTransformer` (`SimpleImputer(strategy="median")` + `StandardScaler()` for numerics; `SimpleImputer(strategy="most_frequent")` + `OneHotEncoder(handle_unknown="ignore")` for categoricals) is nested inside each regression pipeline.
- **Validation Strategy:** Models are compared using **5-fold `KFold` cross-validation** (`shuffle=True, random_state=42`) evaluated with **Mean Absolute Error (MAE)** (`neg_mean_absolute_error`).

In [16]:
reg_df = pd.read_csv("resolution_time_regression.csv")
display(reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [17]:
print(reg_df.info())
print(reg_df.isnull().sum())
print(reg_df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id               800 non-null    str    
 1   complexity_score        784 non-null    float64
 2   team_experience_years   776 non-null    float64
 3   priority                800 non-null    str    
 4   channel                 800 non-null    str    
 5   customer_tenure_months  780 non-null    float64
 6   resolution_time_hours   800 non-null    float64
dtypes: float64(4), str(3)
memory usage: 43.9 KB
None
ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months    20
resolution_time_hours      0
dtype: int64
       complexity_score  team_experience_years  customer_tenure_months  \
count        784.000000             776.000000              780.000000   
mean        

In [24]:
X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
y_reg = reg_df["resolution_time_hours"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

num_cols_reg = X_train_reg.select_dtypes(include=np.number).columns
cat_cols_reg = X_train_reg.select_dtypes(exclude=np.number).columns

num_pipe_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_reg = ColumnTransformer([
    ("num", num_pipe_reg, num_cols_reg),
    ("cat", cat_pipe_reg, cat_cols_reg)
])

## 7. Candidate Model Comparison & Hyperparameter Tuning
We compare **Linear Regression**, **Ridge Regression** (L2-regularized linear model), and **Decision Tree Regressor** (`random_state=42`) across identical 5-fold CV splits.

In [25]:
reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Decision Tree": DecisionTreeRegressor(random_state=42)
}

reg_cv = KFold(n_splits=5, shuffle=True, random_state=42)

reg_results = []
for name, model in reg_models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor_reg),
        ("model", model)
    ])
    scores = cross_validate(
        pipe,
        X_train_reg,
        y_train_reg,
        cv=reg_cv,
        scoring="neg_mean_absolute_error"
    )
    mae_scores = -scores["test_score"]
    reg_results.append({
        "Model": name,
        "Mean MAE": mae_scores.mean(),
        "Std MAE": mae_scores.std()
    })

regression_results = pd.DataFrame(reg_results)
display(regression_results)

,Model,Mean MAE,Std MAE
0,Linear Regression,2.035134,0.171650
1,Ridge Regression,2.035127,0.171404
2,Decision Tree,3.243168,0.051573


### CV Results Analysis & Ridge Tuning
| Model | Mean CV MAE (hours) | Std CV MAE (hours) |
| :--- | :---: | :---: |
| Linear Regression | 2.0351 | 0.1717 |
| **Ridge Regression** | **2.0351** | **0.1714** |
| Decision Tree Regressor | 3.2432 | 0.0516 |

Linear and Ridge Regression achieve ~37% lower MAE (`2.035 hours`) than the unpruned Decision Tree (`3.243 hours`), confirming that resolution time follows a predominantly linear relationship with the features. We tune **Ridge Regression** via `GridSearchCV` across `model__alpha` $\in \{0.01, 0.1, 1, 10, 100\}$.

In [26]:
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor_reg),
    ("model", Ridge())
])

ridge_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

ridge_search = GridSearchCV(
    ridge_pipeline,
    ridge_grid,
    cv=reg_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

ridge_search.fit(X_train_reg, y_train_reg)

print("Best Parameters:", ridge_search.best_params_)
print("Best CV MAE:", -ridge_search.best_score_)

Best Parameters: {'model__alpha': 0.1}
Best CV MAE: 2.035118085555478


## 8. Final Test Evaluation & Outlier/Metric Analysis (Experiment B)
With `alpha = 0.1` (`CV MAE = 2.0351 hours`), we evaluate the final Ridge pipeline once on the holdout test set (`n = 160`).

In [27]:
best_reg_model = ridge_search.best_estimator_
y_pred_reg = best_reg_model.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

print("Test MAE:", mae)
print("Test RMSE:", rmse)

Test MAE: 2.230086144180656
Test RMSE: 3.4027561084563143


### Outliers & Why MAE vs. RMSE May Prefer Different Models
- **Test Results:** **Test MAE = 2.2301 hours**, whereas **Test RMSE = 3.4028 hours** (52.6% larger than MAE).
- **Impact of Outliers:** The target distribution contains extreme right-tail outliers (up to `27.49 hours` vs. a median of `5.39 hours`). Because **MAE** averages absolute residuals ($\frac{1}{n}\sum |y_i - \hat{y}_i|$), every hour of error is weighted linearly, making MAE robust to occasional extreme tickets and reflective of typical median-like prediction accuracy. Conversely, **RMSE** squares residuals before averaging ($\sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}$), disproportionately penalizing large errors on rare, long-running outlier tickets.
- **Model Selection Trade-off:** A model that predicts typical tickets accurately (minimizing MAE) may occasionally miss severe edge cases, yielding a higher RMSE. Conversely, a model regularized or structured to hedge toward extreme values will reduce RMSE at the expense of slightly worse MAE on routine tickets.
- **Justification & Limitations:** Ridge Regression (`alpha=0.1`) generalizes cleanly from CV (`2.035h`) to test (`2.230h`) with low variance and coefficients that resist collinearity among one-hot encoded features. Its primary limitation is linearity—it cannot capture non-linear interaction effects (e.g., high complexity combined with junior team experience) or heavy-tailed skew without target transformation (e.g., `log1p`).

---
# 9. Reflection Questions

### 1. Why stratify the classification split?
The escalation dataset is imbalanced (~78.8% class `0` vs. ~21.2% class `1`). A purely random split risks creating a test set (or CV fold) with disproportionately few positive examples, which distorts F1-score estimates and increases evaluation variance. Stratification enforces identical class proportions across the training set, holdout test set, and every `StratifiedKFold` split.

### 2. Why must preprocessing be inside the pipeline?
Fitting imputers, scalers, or encoders on the full dataset (or across all training folds simultaneously) leaks distributional statistics—such as validation-fold medians, means, variances, and rare category frequencies—into the training process. Placing `ColumnTransformer` inside `Pipeline` ensures preprocessing parameters are learned strictly from each training fold and applied cleanly to validation and test data, preventing optimistic bias and guaranteeing identical transformations at deployment.

### 3. What does fold variability tell you?
The standard deviation across CV folds (`Std F1` or `Std MAE`) measures **model stability and sensitivity to sample composition**. For example, Logistic Regression had a mean CV F1 of `0.3927` with a standard deviation of `0.0627`, indicating moderate sensitivity to which specific minority-class tickets land in each 144-sample validation fold. High fold variability signals potential overfitting, small sample size, or influential outliers, warning against selecting a model based solely on a single split.

### 4. Why should the test set be evaluated only once?
The holdout test set acts as an unbiased proxy for unseen production data. If you repeatedly evaluate models on the test set and adjust features, hyperparameters, or model families based on test scores, information from the test set leaks into design decisions—turning it into a secondary validation set and causing **adaptive overfitting**. Evaluating once at the very end preserves statistical validity.

### 5. What deployment constraints might change your choice?
- **Latency & Memory Footprint:** k-NN stores the entire training set in memory and computes pairwise distances at inference time ($O(N)$ per query), making it unsuitable for high-throughput, real-time ticket routing compared to Logistic Regression or Ridge, which execute a fast dot product in $O(d)$ time.
- **Interpretability & Auditability:** Linear models (Logistic Regression, Ridge) provide transparent feature weights explaining *why* a ticket was escalated or estimated to take longer, which is often required by support managers and SLA governance.
- **Asymmetric Error Costs:** In escalation prediction, missing an urgent escalation (False Negative) is typically far more costly than a false alarm (False Positive). In production, this constraint would require probability threshold tuning or cost-sensitive weighting rather than relying on default accuracy/0.50 cutoffs.